# Item 3 - Decoding baselines for Exp 8

**Question.** Exp 8's evaluation runs Qwen3.5-0.8B with **thinking on** (temperature 0.6, presence penalty 1.5,
32,768 tokens). In that setting ~93 % of MATH-500 answers loop until the token cap, and Astra's best edit (p50c)
mostly removes the loops (item 1). But Qwen's model card says that **0.8B runs in non-thinking mode by default** and
that **in thinking mode it "is more prone to entering thinking loops"**. Exp 8's own `evaluate.py` even has a
non-thinking protocol, with the comment "thinking looped at 0.8B". So: how much of the edit's gain does a
different decoding setting already give, with no edit at all?

| condition | checkpoint | setting |
|---|---|---|
| `base \| think35` | unedited | Exp 8's evaluation (should reproduce ~90 % cut, ~8-14 % acc) |
| `p50c \| think35` | Astra's 50-pass edit | same (should reproduce the post's gain) |
| `base \| nothink35` | unedited | model card's non-thinking benchmark setting, 8,192 tokens |
| `p50c \| nothink35` | Astra's edit | same: does the edit help where the model barely loops? |
| `base \| think_t1` | unedited | model card's thinking setting for text (temperature 1.0) |
| `base \| think35_rep` | unedited | think35 + repetition_penalty 1.1 |

Problems: a fixed random 200 GSM8K test + 200 MATH-500 (the post's benchmarks), 1 sample each, graded with the
release's own grader (`exp8_math/code/mathqa.py`: last `\boxed{}`, math-verify).

**Before you start**
1. Runtime -> Change runtime type -> **A100** (L4 works, ~3x slower; T4 has no bfloat16, not recommended).
2. Nothing to upload: the notebook writes its helper module itself. Results go to `MyDrive/wsp/` if Drive can be
   mounted, else to `/content/wsp` on the VM.
3. Every generation is saved chunk by chunk. If Colab disconnects: re-run the cells up to "Models and
   problems", then the run cell; it continues where it stopped.

Rough cost (A100, not yet measured): a thinking condition writes up to 400 x 32k tokens, about 15-40 min; a
non-thinking one a few minutes. Use `N_GSM = N_MATH = 100` for a first pass.

In [ ]:
# vLLM (generation) + the grader's dependency.  A newer vLLM is needed for Qwen3.5.
!pip install -q -U vllm "math-verify[antlr4_13_2]" datasets
# If loading the model later fails with an unknown architecture (qwen3_5), use the nightly the model card names:
# !pip install -q -U vllm --pre --extra-index-url https://wheels.vllm.ai/nightly
!nvidia-smi --query-gpu=name,memory.total --format=csv

The helper module (the same as `analysis/colab/wsp_colab.py`), written to the VM.

In [ ]:
%%writefile /content/wsp_colab.py
"""Helpers for items 3 and 4 (Colab).  Put this file next to the notebooks in your Drive folder.

Item 3: decoding baselines - is Exp 8's gain something a sampling setting / non-thinking mode already gives?
Item 4: derive the "repetition direction" by hand (diff of means, no LLM), compare it with Astra's, and test edits.

Generation runs in a subprocess (`python wsp_colab.py gen ...`) so that vLLM releases the GPU after each checkpoint.
Every result is written to the Drive folder and every step is resumable.
"""
import argparse
import gc
import glob
import json
import os
import random
import re
import shlex
import shutil
import subprocess
import sys
import time

import numpy as np

BASE_REPO = "Qwen/Qwen3.5-0.8B"
BASE_REV = "2fc06364715b967f1860aea9cf38778875588b17"          # the revision Exp 8 edited
LOGS_URL = "https://github.com/RikiyaT/lm_understand-logs.git"
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."
P = "model.language_model."                                      # tensor-name prefix in the checkpoint

# vLLM sampling settings.  think35 / nothink35 are exactly Exp 8's evaluate.py SAMPLING_THINK35 / SAMPLING_NOTHINK35.
PROTOCOLS = {
    # Exp 8's evaluation: thinking on, the model card's thinking benchmark setting, 32,768 tokens
    "think35": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5, max_tokens=32768),
    # model card, "thinking mode for text tasks"
    "think_t1": dict(thinking=True, temperature=1.0, top_p=0.95, top_k=20, presence_penalty=1.5, max_tokens=32768),
    # think35 + a multiplicative repetition penalty
    "think35_rep": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5,
                        repetition_penalty=1.1, max_tokens=32768),
    # think35 without the presence penalty (model card: "precise coding" setting)
    "think_pp0": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=0.0, max_tokens=32768),
    # the model card's NON-thinking benchmark setting; non-thinking is Qwen3.5-0.8B's default mode
    "nothink35": dict(thinking=False, temperature=0.7, top_p=0.8, top_k=20, presence_penalty=1.5, max_tokens=8192),
    # item 4: practice generations only need the start of each answer (loops start early)
    "think35_8k": dict(thinking=True, temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5, max_tokens=8192),
}


# ===================================================================== setup
q = shlex.quote


def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


def get_logs(dest="/content/lm_understand-logs"):
    """Sparse clone of the logs repo: exp8_math only (code, edits, practice problems)."""
    if not os.path.exists(os.path.join(dest, "exp8_math/code/apply_edit.py")):
        sh(f"git clone --depth 1 --filter=blob:none --sparse {LOGS_URL} {q(dest)}")
        sh(f"cd {q(dest)} && git sparse-checkout set exp8_math")
    return os.path.join(dest, "exp8_math")


def get_base(dest="/content/models/Qwen3.5-0.8B"):
    from huggingface_hub import snapshot_download
    snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=dest)
    return dest


def base_st(base):
    return glob.glob(os.path.join(base, "model*.safetensors"))[0]


def _copy_non_weights(base, out):
    os.makedirs(out, exist_ok=True)
    for f in os.listdir(base):
        src = os.path.join(base, f)
        if os.path.isfile(src) and not f.endswith(".safetensors"):
            shutil.copy(src, os.path.join(out, f))


def official_edit_ckpt(run, exp8, base, out):
    """A full checkpoint with Astra's evaluated edit of `run` (p10c / p25c / p50c), rebuilt bit-exactly and
    hash-checked by the release's apply_edit.py."""
    st = base_st(base)
    target = os.path.join(out, os.path.basename(st))
    if not os.path.exists(os.path.join(out, "DONE")):
        _copy_non_weights(base, out)
        sh(f"{q(sys.executable)} {q(exp8 + '/code/apply_edit.py')} {q(st)} {q(exp8 + '/runs/' + run + '/edit')} {q(target)}")
        open(os.path.join(out, "DONE"), "w").write(run)
    return out


def make_ckpt(base, out, replace):
    """A full checkpoint = base with tensors replaced.  replace: {full tensor name: float array}; stored as bf16."""
    import torch
    from safetensors.torch import load_file, save_file
    if os.path.exists(os.path.join(out, "DONE")):
        return out
    _copy_non_weights(base, out)
    st = base_st(base)
    T = load_file(st)
    for k, v in replace.items():
        assert k in T and tuple(T[k].shape) == tuple(v.shape), k
        T[k] = torch.as_tensor(np.asarray(v, dtype=np.float32)).to(torch.bfloat16).contiguous()
    save_file(T, os.path.join(out, os.path.basename(st)), metadata={"format": "pt"})
    json.dump(sorted(replace), open(os.path.join(out, "DONE"), "w"))
    del T
    gc.collect()
    return out


# ===================================================================== grading (the release's own grader)
_GRADER = []


def grader(exp8):
    """exp8_math/code/mathqa.py: extract() = the last \\boxed{} after </think>, is_correct() = math-verify."""
    if not _GRADER:
        stub = os.path.join(os.path.dirname(os.path.abspath(__file__)), "_stubs")
        os.makedirs(stub, exist_ok=True)
        open(os.path.join(stub, "mcqa.py"), "w").write(
            "def loop_signals(text):\n    return {'repetitive_end': False}\n")   # only used by a function we skip
        for p in (stub, os.path.join(exp8, "code")):
            if p not in sys.path:
                sys.path.insert(0, p)
        import mathqa
        _GRADER.append(mathqa)
    return _GRADER[0]


# ===================================================================== problems
def _write_jsonl(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    os.replace(tmp, path)


def read_jsonl(path):
    return [json.loads(l) for l in open(path)]


def load_test(path, n_gsm=200, n_math=200, seed=0):
    """A fixed random subset of GSM8K test and MATH-500 (the benchmarks of the post), saved once to `path`."""
    if os.path.exists(path):
        return read_jsonl(path)
    from datasets import load_dataset
    rng = random.Random(seed)
    g = load_dataset("openai/gsm8k", "main", split="test")
    m = load_dataset("HuggingFaceH4/MATH-500", split="test")
    rows = []
    for i in sorted(rng.sample(range(len(g)), n_gsm)):
        rows.append(dict(id=f"gsm8k_test_{i}", source="gsm8k", problem=g[i]["question"],
                         answer=g[i]["answer"].split("####")[-1].strip().replace(",", "")))
    for i in sorted(rng.sample(range(len(m)), n_math)):
        rows.append(dict(id=m[i]["unique_id"], source="math500", problem=m[i]["problem"], answer=m[i]["answer"]))
    _write_jsonl(path, rows)
    return rows


def load_practice(path, exp8, n_gsm=150, n_math=150, seed=1):
    """Practice problems Astra could use (GSM8K train, MATH train) - never the test sets."""
    if os.path.exists(path):
        return read_jsonl(path)
    rng = random.Random(seed)
    rows = []
    for name, n in (("gsm8k_train", n_gsm), ("math_train", n_math)):
        allr = read_jsonl(os.path.join(exp8, "practice", f"{name}.jsonl"))
        rows += [dict(id=r["id"], source=r["source"], problem=r["problem"], answer=r["answer"]) for r in rng.sample(allr, n)]
    _write_jsonl(path, rows)
    return rows


def prompt_text(tok, problem, thinking):
    return tok.apply_chat_template([{"role": "user", "content": problem + "\n" + INSTRUCTION}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=thinking)


# ===================================================================== generation (vLLM, in a subprocess)
def generate(ckpt, problems_path, protocol, out_path, n=1, gpu_mem=0.85, chunk=100):
    """Resumable: answers are appended chunk by chunk to out_path; problems already there are skipped."""
    done = {r["id"] for r in read_jsonl(out_path)} if os.path.exists(out_path) else set()
    todo = [r for r in read_jsonl(problems_path) if r["id"] not in done]
    if not todo:
        print(f"[skip] {out_path}: complete")
        return out_path
    t = time.time()
    sh(f"{q(sys.executable)} {q(os.path.abspath(__file__))} gen --ckpt {q(ckpt)} --problems {q(problems_path)} "
       f"--protocol {protocol} --out {q(out_path)} --n {n} --gpu_mem {gpu_mem} --chunk {chunk}")
    print(f"[done] {out_path} in {(time.time() - t) / 60:.1f} min")
    return out_path


def _gen_main(a):
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    cfg = dict(PROTOCOLS[a.protocol])
    thinking = cfg.pop("thinking")
    tok = AutoTokenizer.from_pretrained(a.ckpt)
    done = {r["id"] for r in read_jsonl(a.out)} if os.path.exists(a.out) else set()
    rows = [r for r in read_jsonl(a.problems) if r["id"] not in done]
    llm = None
    common = dict(model=a.ckpt, dtype="bfloat16", gpu_memory_utilization=a.gpu_mem,
                  max_model_len=cfg["max_tokens"] + 2048, seed=0)
    for extra in ({"language_model_only": True}, {"limit_mm_per_prompt": {"image": 0, "video": 0}}, {}):
        try:
            llm = LLM(**common, **extra)
            print("vLLM started with", extra or "no extra options", flush=True)
            break
        except (TypeError, ValueError) as e:      # option unknown to this vLLM version
            print("vLLM option", extra, "refused:", str(e)[:200], flush=True)
    if llm is None:
        raise RuntimeError("vLLM could not load the model; install a newer vLLM (see the install cell)")
    sp = SamplingParams(n=a.n, seed=0, **cfg)
    for s in range(0, len(rows), a.chunk):
        part = rows[s:s + a.chunk]
        outs = llm.generate([prompt_text(tok, r["problem"], thinking) for r in part], sp)
        with open(a.out, "a") as f:
            for r, o in zip(part, outs):
                for k, x in enumerate(o.outputs):
                    f.write(json.dumps(dict(id=r["id"], source=r["source"], answer=r["answer"], sample=k,
                                            text=x.text, finish=x.finish_reason, n_tokens=len(x.token_ids))) + "\n")
        print(f"  {min(s + a.chunk, len(rows))}/{len(rows)} problems", flush=True)


# ===================================================================== scoring
NGRAM = 8


def loop_stats(text):
    """Word-level loop measures (same as item 1).
    dup_tail: share of the last 500 word-8-grams seen earlier; onset_char: where the text turns into a loop (the
    earliest word from which >= 90% of 8-grams are repeats, the loop lasting >= 200 words), -1 if never;
    onset_rel: the same as a share of the words; period: words between the last 8-gram and its previous occurrence."""
    spans = [m.start() for m in re.finditer(r"\S+", text)]
    w = text.split()
    n = len(w) - NGRAM + 1
    tail = text[-2000:]
    ws_tail = len(tail) > 0 and sum(c.isspace() for c in tail) / len(tail) >= 0.9
    if n <= 0:
        return dict(words=len(w), dup_tail=0.0, onset_char=-1, onset_rel=np.nan, period=np.nan, ws_tail=ws_tail)
    seen, last = set(), {}
    dup = np.zeros(n, dtype=np.int8)
    dists = []
    for i in range(n):
        g = tuple(w[i:i + NGRAM])
        if g in seen:
            dup[i] = 1
            dists.append((i, i - last[g]))
        seen.add(g)
        last[g] = i
    suf = np.cumsum(dup[::-1])[::-1] / np.arange(n, 0, -1)
    ok = np.where((suf >= 0.9) & (np.arange(n) < n - 200))[0]
    onset = int(ok[0]) if len(ok) else -1
    after = [d for i, d in dists if onset >= 0 and i >= onset]
    return dict(words=len(w), dup_tail=float(dup[-500:].mean()), onset_char=spans[onset] if onset >= 0 else -1,
                onset_rel=onset / n if onset >= 0 else np.nan,
                period=float(np.median(after)) if after else np.nan, ws_tail=ws_tail)


def score_file(path, exp8, cache=True):
    """One row per answer: correct, cut (finish_reason == length), loop measures, kind."""
    import pandas as pd
    cp = path.replace(".jsonl", ".scored.csv")
    if cache and os.path.exists(cp) and os.path.getmtime(cp) > os.path.getmtime(path):
        return pd.read_csv(cp)
    Q = grader(exp8)
    recs = []
    for r in read_jsonl(path):
        pred, nbox = Q.extract(r["text"])
        st = loop_stats(r["text"])
        cut = r["finish"] == "length"
        kind = "finished" if not cut else "cut: whitespace" if st["ws_tail"] else \
            "cut: loop" if st["dup_tail"] >= 0.8 else "cut: no loop"
        recs.append(dict(id=r["id"], source=r["source"], sample=r["sample"], correct=int(Q.is_correct(pred, r["answer"])),
                         pred=pred, cut=int(cut), n_tokens=r["n_tokens"], closed_think=int("</think>" in r["text"]),
                         kind=kind, **st))
    df = pd.DataFrame(recs)
    if cache:
        df.to_csv(cp, index=False)
    return df


def summarize(dfs, baseline):
    """dfs: {condition: scored DataFrame}.  Accuracy with SE, cut / loop shares, tokens, paired gain vs baseline."""
    import pandas as pd
    b = dfs[baseline].set_index(["id", "sample"])["correct"]
    rows = []
    for name, d in dfs.items():
        x = d.set_index(["id", "sample"])["correct"]
        j = pd.concat([x, b], axis=1, keys=["c", "b"]).dropna()
        diff = j.c - j.b
        row = dict(condition=name, n=len(d), acc=d.correct.mean(), se=d.correct.std() / np.sqrt(len(d)),
                   cut=d.cut.mean(), loop=(d.kind == "cut: loop").mean(), mean_tokens=d.n_tokens.mean(),
                   gain_vs_base=diff.mean(), gain_se=diff.std() / np.sqrt(len(diff)) if len(diff) > 1 else np.nan,
                   rescued=int(((j.c == 1) & (j.b == 0)).sum()), lost=int(((j.c == 0) & (j.b == 1)).sum()))
        for s in sorted(d.source.unique()):
            row[f"acc_{s}"] = d[d.source == s].correct.mean()
        row["acc_if_finished"] = d[d.cut == 0].correct.mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index("condition")


# ===================================================================== item 4: contexts and activations
def build_contexts(gen_path, scored, seed=0):
    """From base-model practice generations: contexts that end INSIDE a loop ("rep") and matched contexts from the
    same answers that end well before the loop starts ("prod"); plus contexts from answers that finished
    ("fin", an unmatched productive control).  Each keeps the next 64 tokens of the real text (for the log-prob test)."""
    rng = random.Random(seed)
    gens = {(r["id"], r["sample"]): r for r in read_jsonl(gen_path)}
    ctx = []
    for _, s in scored.iterrows():
        g = gens[(s["id"], s["sample"])]
        t = g["text"]
        if s["kind"] == "cut: loop" and s["onset_char"] >= 1500:
            o = int(s["onset_char"])
            per = s["period"] if s["period"] == s["period"] else 20
            into = min(len(t) - 400, o + max(300, int(3 * per * 6)))       # ~3 loop periods (~6 chars / word)
            prod = int(o * rng.uniform(0.3, 0.8))
            for kind, end in (("rep", into), ("prod", prod)):
                ctx.append(dict(kind=kind, gid=f"{s['id']}#{s['sample']}", id=s["id"], prefix=t[:end],
                                cont=t[end:end + 600]))
        elif s["kind"] == "finished" and s["closed_think"]:
            close = t.find("</think>")
            if close > 1500:
                end = int(close * rng.uniform(0.2, 0.7))
                ctx.append(dict(kind="fin", gid=f"{s['id']}#{s['sample']}", id=s["id"], prefix=t[:end],
                                cont=t[end:end + 600]))
    return ctx


def tokenize_contexts(ctx, tok, problems, thinking=True, max_ctx_tokens=8000, cont_tokens=64):
    out = []
    pmap = {p["id"]: p for p in problems}
    for c in ctx:
        head = prompt_text(tok, pmap[c["id"]]["problem"], thinking)
        ids = tok(head + c["prefix"], add_special_tokens=False).input_ids
        cont = tok(c["cont"], add_special_tokens=False).input_ids[:cont_tokens]
        if len(ids) <= max_ctx_tokens and len(cont) == cont_tokens:
            out.append(dict(c, ids=ids, cont_ids=cont))
    return out


def load_hf(base, device="cuda"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(base)
    m = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16 if device == "cuda" else torch.float32)
    return m.to(device).eval(), tok


def collect_acts(model, ctxs, layers=tuple(range(8, 23)), neuron_layers=(15, 19), K=32, log_every=50):
    """Per context: residual stream after each layer in `layers` (hidden_states[L+1]) and the MLP neuron
    activations (input of down_proj) of `neuron_layers`, both averaged over the context's last K tokens."""
    import torch
    dev = next(model.parameters()).device
    store = {}
    hooks = [model.get_submodule(f"model.layers.{L}.mlp.down_proj").register_forward_hook(
        lambda mod, inp, out, L=L: store.__setitem__(L, inp[0][0, -K:].float().mean(0).cpu().numpy()))
        for L in neuron_layers]
    H = {L: [] for L in layers}
    N = {L: [] for L in neuron_layers}
    t = time.time()
    try:
        with torch.no_grad():
            for i, c in enumerate(ctxs):
                o = model(torch.tensor([c["ids"]], device=dev), output_hidden_states=True, use_cache=False)
                for L in layers:
                    H[L].append(o.hidden_states[L + 1][0, -K:].float().mean(0).cpu().numpy())
                for L in neuron_layers:
                    N[L].append(store[L])
                del o
                if (i + 1) % log_every == 0:
                    print(f"  {i + 1}/{len(ctxs)} contexts, {time.time() - t:.0f}s", flush=True)
    finally:
        for h in hooks:
            h.remove()
    return {L: np.stack(v) for L, v in H.items()}, {L: np.stack(v) for L, v in N.items()}


def diff_direction(X, kinds, pos="rep", neg="prod"):
    kinds = np.asarray(kinds)
    d = X[kinds == pos].mean(0) - X[kinds == neg].mean(0)
    return d / np.linalg.norm(d)


def astra_directions(exp8, base, work):
    """Residual directions of Astra's p10c (one per layer, 8-22) and p50c (layers 15, 19) edits: top left singular
    vector of (edited - base) for each tensor (rank one, see item 2)."""
    import torch
    from safetensors.torch import load_file
    out = {}
    W0 = None
    for run in ("p10c", "p50c"):
        d = os.path.join(work, "astra_edits", run)
        if not os.path.exists(d):
            sh(f"{q(sys.executable)} {q(exp8 + '/code/apply_edit.py')} {q(base_st(base))} {q(exp8 + '/runs/' + run + '/edit')} {q(d)}")
        if W0 is None:
            W0 = load_file(base_st(base))
        for f in sorted(glob.glob(f"{d}/*.npy")):
            n = os.path.basename(f)[:-4]
            if run == "p10c" and "mlp.down_proj" not in n:
                continue                                      # p10c: one direction per layer, shared by its tensors
            D = np.load(f).astype(np.float64) - W0[n].float().numpy().astype(np.float64)
            u = np.linalg.svd(D, full_matrices=False)[0][:, 0]
            out[(run, int(n.split("layers.")[1].split(".")[0]))] = u
    return out


def top_k_edit(W0, u, act_mean, k=64, alpha=6.0):
    """Astra's p50c form: W = W0 - alpha * u (mask * (u^T W0)), on the k neurons that write the most along u
    in the measured contexts (score_j = |mean activation_j * u . W0[:, j]|).  W0: (1024, 3584) float64."""
    w = u @ W0
    score = np.abs(act_mean * w)
    cols = np.argsort(-score)[:k]
    mask = np.zeros_like(w)
    mask[cols] = 1
    return W0 - alpha * np.outer(u, mask * w), cols


def random_control(W0, u, act_mean, k, alpha, seed=0):
    """A random unit direction orthogonal to u, same selection rule, scaled so that |dW| matches the u-edit's."""
    r = np.random.default_rng(seed).standard_normal(len(u))
    r -= (r @ u) * u
    r /= np.linalg.norm(r)
    We, _ = top_k_edit(W0, u, act_mean, k, alpha)
    Wr, cols = top_k_edit(W0, r, act_mean, k, 1.0)
    s = np.linalg.norm(We - W0) / np.linalg.norm(Wr - W0)
    return W0 + s * (Wr - W0), cols, r


def logprob_test(model, ctxs, edits=None):
    """Mean log-prob per token of each context's real 64-token continuation, with optional in-memory edits
    {layer: down_proj W (1024, 3584)}.  Teacher-forced: one forward pass per context."""
    import torch
    dev = next(model.parameters()).device
    saved = {}
    try:
        for L, W in (edits or {}).items():
            p = model.get_submodule(f"model.layers.{L}.mlp.down_proj").weight
            saved[L] = p.data.clone()
            p.data = torch.as_tensor(W, dtype=p.dtype, device=dev)
        res = []
        with torch.no_grad():
            for c in ctxs:
                ids = torch.tensor([c["ids"] + c["cont_ids"]], device=dev)
                lp = torch.log_softmax(model(ids, use_cache=False).logits[0, len(c["ids"]) - 1:-1].float(), -1)
                res.append(lp.gather(1, ids[0, len(c["ids"]):, None]).mean().item())
        return np.array(res)
    finally:
        for L, W in saved.items():
            model.get_submodule(f"model.layers.{L}.mlp.down_proj").weight.data = W


# ===================================================================== CLI (the generation subprocess)
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["gen"])
    ap.add_argument("--ckpt")
    ap.add_argument("--problems")
    ap.add_argument("--protocol")
    ap.add_argument("--out")
    ap.add_argument("--n", type=int, default=1)
    ap.add_argument("--gpu_mem", type=float, default=0.85)
    ap.add_argument("--chunk", type=int, default=100)
    _gen_main(ap.parse_args())

In [ ]:
# Results go to Google Drive when it can be mounted (survives the runtime); otherwise to /content/wsp on the VM
# (e.g. in VS Code's Colab extension, where drive.mount may not work): download them before the runtime ends.
USE_DRIVE = True
import os, sys, importlib
WORK = '/content/wsp'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK = '/content/drive/MyDrive/wsp'
    except Exception as e:
        print('Google Drive not available here:', type(e).__name__, str(e)[:200])
print('results folder:', WORK)
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, '/content')
import wsp_colab as W; importlib.reload(W)
os.makedirs(f'{WORK}/item3', exist_ok=True)

## Configuration

In [ ]:
N_GSM, N_MATH = 200, 200          # fixed random test subset (saved once to WORK/problems_test.jsonl, shared with item 4)
BASELINE = 'base | think35'
CONDITIONS = [                    # (name, checkpoint, protocol in W.PROTOCOLS)
    ('base | think35',     'base', 'think35'),
    ('p50c | think35',     'p50c', 'think35'),
    ('base | nothink35',   'base', 'nothink35'),
    ('p50c | nothink35',   'p50c', 'nothink35'),
    ('base | think_t1',    'base', 'think_t1'),
    ('base | think35_rep', 'base', 'think35_rep'),
    # optional:
    # ('base | think_pp0', 'base', 'think_pp0'),
    # ('p10c | think35',   'p10c', 'think35'),
    # ('p25c | think35',   'p25c', 'think35'),
]
for k, v in W.PROTOCOLS.items(): print(f'{k:12s}', v)

## Models and problems
The edited checkpoints are rebuilt bit-exactly from the public base weights by the release's `apply_edit.py` (it checks every tensor's sha256).

In [ ]:
exp8 = W.get_logs()                       # sparse clone of lm_understand-logs/exp8_math
base = W.get_base()                       # Qwen/Qwen3.5-0.8B at the revision Exp 8 edited
CKPT = {'base': base}
for run in sorted({c[1] for c in CONDITIONS} - {'base'}):
    CKPT[run] = W.official_edit_ckpt(run, exp8, base, f'/content/ckpts/{run}')
problems_path = f'{WORK}/problems_test.jsonl'
probs = W.load_test(problems_path, N_GSM, N_MATH)
print(len(probs), 'problems;', sum(p['source'] == 'gsm8k' for p in probs), 'GSM8K')

## Smoke test (1-2 min)
Eight problems, non-thinking, to check that vLLM loads the model and the grader works.

In [ ]:
smoke = '/content/smoke_problems.jsonl'
W._write_jsonl(smoke, probs[:4] + probs[-4:])
if os.path.exists('/content/smoke_out.jsonl'): os.remove('/content/smoke_out.jsonl')
W.generate(base, smoke, 'nothink35', '/content/smoke_out.jsonl')
W.score_file('/content/smoke_out.jsonl', exp8, cache=False)[['id', 'correct', 'pred', 'cut', 'n_tokens']]

## Run every condition (resumable)

In [ ]:
def out_path(name):
    return f"{WORK}/item3/{name.replace(' | ', '__')}.jsonl"

for name, ck, proto in CONDITIONS:
    print('=' * 20, name)
    W.generate(CKPT[ck], problems_path, proto, out_path(name))

## Results

In [ ]:
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
dfs = {name: W.score_file(out_path(name), exp8) for name, _, _ in CONDITIONS if os.path.exists(out_path(name))}
S = W.summarize(dfs, BASELINE)
S.to_csv(f'{WORK}/item3/summary.csv')
S[['n', 'acc', 'se', 'acc_gsm8k', 'acc_math500', 'cut', 'loop', 'mean_tokens', 'acc_if_finished',
   'gain_vs_base', 'gain_se', 'rescued', 'lost']]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
S[['acc_gsm8k', 'acc_math500']].plot.barh(ax=ax[0]); ax[0].set_xlabel('accuracy'); ax[0].invert_yaxis()
kinds = pd.DataFrame({n: d.kind.value_counts(normalize=True) for n, d in dfs.items()}).T.fillna(0)
cols = [c for c in ['finished', 'cut: loop', 'cut: whitespace', 'cut: no loop'] if c in kinds]
kinds[cols].plot.barh(stacked=True, ax=ax[1], color=['#4c9a5b', '#c0504d', '#e3a33b', '#8c8c8c'][:len(cols)])
ax[1].set_xlabel('share of answers'); ax[1].invert_yaxis(); ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(f'{WORK}/item3/fig_item3.png', dpi=140)

In [ ]:
# Where does each condition's gain over the baseline come from?  (paired by problem)
b = dfs[BASELINE].set_index(['id', 'sample'])
for name, d in dfs.items():
    if name == BASELINE: continue
    j = d.set_index(['id', 'sample']).join(b, rsuffix='_b', how='inner')
    gained = (j.correct == 1) & (j.correct_b == 0)
    print(f'{name:22s} gained {gained.sum():3d}, of which the baseline answer was cut: {(gained & (j.cut_b == 1)).sum():3d};'
          f'  lost {((j.correct == 0) & (j.correct_b == 1)).sum():3d}')

## How to read it
- **`base | nothink35` close to `p50c | think35`**: the edit mostly recovers what the model's default mode already
  gives; the "math gain" is a fix for a thinking-mode failure that the model card documents.
- **`base | think_t1` or `base | think35_rep` already removes most loops**: a sampling setting does the edit's job.
- **`p50c | nothink35` vs `base | nothink35`**: the edit's effect where loops are rare - like the Gemma follow-up,
  expect little.
- `acc_if_finished` and the "gained ... cut" line test item 1's claim on fresh samples.